In [1]:
print('hi')

hi


# Load the Data

In [140]:
from langchain.document_loaders import PyPDFLoader

In [141]:
loader = PyPDFLoader('EMPLOYEE_AGREEMENT.pdf')

In [142]:
loaded = loader.load()

In [143]:
loaded

[Document(metadata={'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page': 0, 'page_label': '1'}, page_content='EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT\nEXHIBIT 10.1\nEMPLOYEE AGREEMENT\nTHIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,\nInc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,\nSuite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in\nits entirety the employee agreement dated August 15, 2014 between Employee and the Company. \nWHEREAS, the Company wishes to procure the services of Employee under the terms and\nconditions set forth and Employee wishes to be employed 

In [7]:
loaded[0].page_content

'EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT\nEXHIBIT 10.1\nEMPLOYEE AGREEMENT\nTHIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,\nInc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,\nSuite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in\nits entirety the employee agreement dated August 15, 2014 between Employee and the Company. \nWHEREAS, the Company wishes to procure the services of Employee under the terms and\nconditions set forth and Employee wishes to be employed on these terms and conditions.\nWHEREAS, the parties to this Employee Agreement wish to enter into a written expression of their\nrelationship as Employer and Employee.\nTHEREFORE, in consideration of the agreements contained in this Employee Agreement, the\nparties, intending to be legally bound, agree as follows:\nARTICLE 1\nEmployment\n1.1. Employment. The Company agrees to employ Emp

In [8]:
len(loaded)

13

# Chuckning 

In [9]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


In [144]:
text_spiltter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

In [145]:
chunks = text_spiltter.split_documents(loaded)

In [146]:
len(chunks)

41

In [19]:
print(chunks[0].page_content)

EX-10.1 3 smtp_ex10z1.htm EMPLOYEE AGREEMENT
EXHIBIT 10.1
EMPLOYEE AGREEMENT
THIS EMPLOYEE AGREEMENT made as of September___, 2014, by and between SharpSpring,
Inc., a Delaware corporation (the “Company”), whose principal place of business is at 802 NW 5th Avenue,
Suite 100, Gainesville FL 32601; and Richard Carlson (“Employee”). This Employee Agreement replaces in
its entirety the employee agreement dated August 15, 2014 between Employee and the Company. 
WHEREAS, the Company wishes to procure the services of Employee under the terms and
conditions set forth and Employee wishes to be employed on these terms and conditions.
WHEREAS, the parties to this Employee Agreement wish to enter into a written expression of their
relationship as Employer and Employee.
THEREFORE, in consideration of the agreements contained in this Employee Agreement, the
parties, intending to be legally bound, agree as follows:
ARTICLE 1
Employment


# embedding 

In [20]:
from dotenv import load_dotenv
load_dotenv()

True

In [150]:
import os
cohere_api_key = os.getenv('cohere_api_key')


In [23]:
os.environ['cohere_api_key'] = cohere_api_key

In [24]:
from langchain_cohere import CohereEmbeddings

embedding = CohereEmbeddings(
    model = "embed-english-v3.0"
)

# Vectore Database

In [25]:
from langchain.vectorstores import FAISS

In [26]:
vectorestore = FAISS.from_documents(chunks,embedding)

In [74]:
retriver = vectorestore.as_retriever()

In [71]:
vectorestore.similarity_search("what is base compensation",k=10)

[Document(id='312105b2-79f8-4d1d-b0b9-df4567e41f44', metadata={'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page': 1, 'page_label': '2'}, page_content='ARTICLE 3\nPlace of Employment\n3.1. Place of Employment.   Employee shall perform his duties under this Employee Agreement at\n802 NW 5th Avenue, Suite 100, Gainesville FL 32601.\nARTICLE 4\nCompensation of Employee\n4.1. Base Compensation.  For all services rendered by Employee under this Employee Agreement, the\nCompany agrees to pay Employee the rate of $14,583 per month (the “base salary”), which shall be payable\nto Employee not less frequently than bi-monthly, or as is consistent with the Company’s practice for its other\nemployees.  \n4.2. Other Compen

# llm 

In [72]:
from langchain_cohere import ChatCohere
llm = ChatCohere()

In [80]:
from langchain.memory import ConversationBufferMemory


In [96]:
from langchain.chains import ConversationalRetrievalChain
chain = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = vectorestore.as_retriever(),
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
)

In [97]:
res = chain.invoke("Give me summary of the document")

In [98]:
print(res['answer'])

This document is an **Employee Agreement** between **SharpSpring, Inc.** and **Richard Carlson**, outlining key terms and obligations related to his employment. Here’s a summary of the main points:

1. **Confidentiality Obligations**:  
   - The employee (Richard Carlson) must keep all confidential information of the company and its affiliates confidential and not use it against the company's interests.  

2. **Post-Employment Obligations**:  
   - After termination, the employee must return all company records, files, and documents related to customers, suppliers, duties, or the company’s business.  
   - For three years post-termination, the employee must inform future employers, business partners, or colleagues about the existence of specific restrictive covenants (Sections 9.1 and 9.2) and provide a copy if requested.  

3. **Termination Provisions**:  
   - The company may terminate employment for "cause," defined as willful malfeasance, misfeasance, nonfeasance, gross negligence,

In [99]:
res = chain.invoke("what is Gen AI")

In [100]:
res['answer']

'La IA generativa (Inteligencia Artificial Generativa) es un tipo de tecnología de inteligencia artificial que se enfoca en la creación de contenido nuevo y original, como imágenes, texto, música o incluso código de programación, a partir de datos de entrada. A diferencia de los modelos de IA tradicionales que se utilizan principalmente para clasificación, predicción o análisis, la IA generativa tiene la capacidad de generar salida creativa y diversa.\n\nEsta tecnología se basa en redes neuronales profundas, especialmente en arquitecturas como las Redes Adversariales Generativas (GAN, por sus siglas en inglés) y los modelos de Transformadores. Aquí hay una breve explicación de cada uno:\n\n1. **Redes Adversariales Generativas (GAN)**: Consisten en dos redes neuronales que compiten entre sí: un generador y un discriminador. El generador crea nuevos datos (como imágenes) y el discriminador evalúa si estos datos son reales o falsos. A través de este proceso de competencia, el generador me

In [149]:
groq_api_key = os.getenv('groq_api_key')


In [102]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model = 'llama-3.1-8b-instant',
    api_key=groq_api_key
)

In [103]:
from langchain.chains import ConversationalRetrievalChain
chain = ConversationalRetrievalChain.from_llm(
    llm = llm,
    retriever = vectorestore.as_retriever(),
    memory = ConversationBufferMemory(memory_key='chat_history', return_messages=True)
)

In [104]:
res = chain.invoke("what is Gen AI")

In [106]:
res['answer']

"I don't know."

In [107]:
res = chain.invoke("what is base compensation")

In [109]:
print(res['answer'])

According to ARTICLE 4 of the Employee Agreement, "Base Compensation" is defined as $14,583 per month, payable to the Employee not less frequently than bi-monthly.


# Load the Document as fully string -> text`

In [110]:
from langchain.document_loaders import PyPDFLoader

In [111]:
loader = PyPDFLoader('Kabir_Sk_Resume.pdf')

In [112]:
loaded = loader.load()

In [113]:
loaded

[Document(metadata={'producer': 'pdfTeX-1.40.26', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-06-13T06:47:22+00:00', 'author': '', 'keywords': '', 'moddate': '2025-06-13T06:47:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.26 (TeX Live 2024) kpathsea version 6.4.0', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'Kabir_Sk_Resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Kabir Sk +91-8583017348\nAI Developer | Backend Engineer | GenAI Developer\nLeetcode | GeeksForGeeks kabirsk58694@gmail.com\nGithub | LinkedIn Kolkata, West Bengal\nUniversity of Engineering and Management, Kolkata\nEducation\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech. (CSE - AIML ) University of Engineering and Management, Kolkata 8.89 2020-2024\nSenior Secondary | XII Barrackpore AB Model High School/WBCHSE Board 70.0% 2020\nExperience\n• Infosys Limited Sept 2024 – Present\nSpecialist Programmer Hyderabad, Telangana,

In [115]:
text = ""

for doc in loaded:
    text+=doc.page_content + "\n\n"

In [116]:
print(text)

Kabir Sk +91-8583017348
AI Developer | Backend Engineer | GenAI Developer
Leetcode | GeeksForGeeks kabirsk58694@gmail.com
Github | LinkedIn Kolkata, West Bengal
University of Engineering and Management, Kolkata
Education
Degree/Certificate Institute/Board CGPA/Percentage Year
B.Tech. (CSE - AIML ) University of Engineering and Management, Kolkata 8.89 2020-2024
Senior Secondary | XII Barrackpore AB Model High School/WBCHSE Board 70.0% 2020
Experience
• Infosys Limited Sept 2024 – Present
Specialist Programmer Hyderabad, Telangana, India
– Designed and deployed ameta-agent orchestration layerwith aSupervisor Agentto manage multi-agent workflows
(including ReAct-based agents), resulting in a35% boost in agent utilizationand 25% reduction in response latency
for complex tasks.
– Developedanenterprise-grade RAG(Retrieval-AugmentedGeneration) systemenablingsemanticsearchover PDF,
PPT, Excel, and image files, reducing document retrieval time by65% and improving answer relevance by80%.
– Spea

# Chunking

In [117]:
text_spiltter = RecursiveCharacterTextSplitter(
    chunk_size = 100,
    chunk_overlap = 10
)

In [118]:
chunks = text_spiltter.split_text(text)

In [119]:
len(chunks)

59

In [122]:
chunks

['Kabir Sk +91-8583017348\nAI Developer | Backend Engineer | GenAI Developer',
 'Leetcode | GeeksForGeeks kabirsk58694@gmail.com\nGithub | LinkedIn Kolkata, West Bengal',
 'University of Engineering and Management, Kolkata\nEducation',
 'Education\nDegree/Certificate Institute/Board CGPA/Percentage Year',
 'B.Tech. (CSE - AIML ) University of Engineering and Management, Kolkata 8.89 2020-2024',
 'Senior Secondary | XII Barrackpore AB Model High School/WBCHSE Board 70.0% 2020\nExperience',
 '• Infosys Limited Sept 2024 – Present\nSpecialist Programmer Hyderabad, Telangana, India',
 '– Designed and deployed ameta-agent orchestration layerwith aSupervisor Agentto manage multi-agent',
 'workflows',
 '(including ReAct-based agents), resulting in a35% boost in agent utilizationand 25% reduction in',
 'in response latency',
 'for complex tasks.',
 '– Developedanenterprise-grade RAG(Retrieval-AugmentedGeneration) systemenablingsemanticsearchover',
 'PDF,',
 'PPT, Excel, and image files, reduci

# vectore store

In [124]:
vectorestore = FAISS.from_texts(chunks, embedding)

In [127]:
vectorestore.similarity_search('what is weather today')

[Document(id='9e105a4c-a039-41a3-986f-0e8b1ecf4230', metadata={}, page_content='curated insights—weather,'),
 Document(id='03058977-a8f8-4856-9a73-07fbf9ee7a7c', metadata={}, page_content='Systems'),
 Document(id='5c2f281e-6f31-473a-bfdd-ce0cff75efeb', metadata={}, page_content='–Frameworks - Libraries:Streamlit, FastAPI, Flask, Pandas, Numpy\n–Web: HTML, CSS'),
 Document(id='d9787ae0-8b88-47d4-beb9-be54f7cf0b68', metadata={}, page_content='Education\nDegree/Certificate Institute/Board CGPA/Percentage Year')]

In [128]:
from langchain.docstore.document import Document


In [132]:
text

'Kabir Sk +91-8583017348\nAI Developer | Backend Engineer | GenAI Developer\nLeetcode | GeeksForGeeks kabirsk58694@gmail.com\nGithub | LinkedIn Kolkata, West Bengal\nUniversity of Engineering and Management, Kolkata\nEducation\nDegree/Certificate Institute/Board CGPA/Percentage Year\nB.Tech. (CSE - AIML ) University of Engineering and Management, Kolkata 8.89 2020-2024\nSenior Secondary | XII Barrackpore AB Model High School/WBCHSE Board 70.0% 2020\nExperience\n• Infosys Limited Sept 2024 – Present\nSpecialist Programmer Hyderabad, Telangana, India\n– Designed and deployed ameta-agent orchestration layerwith aSupervisor Agentto manage multi-agent workflows\n(including ReAct-based agents), resulting in a35% boost in agent utilizationand 25% reduction in response latency\nfor complex tasks.\n– Developedanenterprise-grade RAG(Retrieval-AugmentedGeneration) systemenablingsemanticsearchover PDF,\nPPT, Excel, and image files, reducing document retrieval time by65% and improving answer releva

In [133]:
from importlib import metadata


documents = [Document(page_content=doc,metadata={'source':"kabir_sk_reums"}) for doc in chunks]

In [134]:
documents

[Document(metadata={'source': 'kabir_sk_reums'}, page_content='Kabir Sk +91-8583017348\nAI Developer | Backend Engineer | GenAI Developer'),
 Document(metadata={'source': 'kabir_sk_reums'}, page_content='Leetcode | GeeksForGeeks kabirsk58694@gmail.com\nGithub | LinkedIn Kolkata, West Bengal'),
 Document(metadata={'source': 'kabir_sk_reums'}, page_content='University of Engineering and Management, Kolkata\nEducation'),
 Document(metadata={'source': 'kabir_sk_reums'}, page_content='Education\nDegree/Certificate Institute/Board CGPA/Percentage Year'),
 Document(metadata={'source': 'kabir_sk_reums'}, page_content='B.Tech. (CSE - AIML ) University of Engineering and Management, Kolkata 8.89 2020-2024'),
 Document(metadata={'source': 'kabir_sk_reums'}, page_content='Senior Secondary | XII Barrackpore AB Model High School/WBCHSE Board 70.0% 2020\nExperience'),
 Document(metadata={'source': 'kabir_sk_reums'}, page_content='• Infosys Limited Sept 2024 – Present\nSpecialist Programmer Hyderabad,

In [135]:
vectorestore.save_local('faiss_db')

In [136]:
vectorestore = vectorestore.load_local('faiss_db',embeddings=embedding, allow_dangerous_deserialization=True)

In [ ]:
vectorestore.similarity_search(
    'who is kabir',k=10
)

[Document(id='e6eeffce-0fce-4f12-93c4-271c9ff39437', metadata={}, page_content='Kabir Sk +91-8583017348\nAI Developer | Backend Engineer | GenAI Developer'),
 Document(id='01e4a171-ab04-4200-9b15-16ef8172663d', metadata={}, page_content='Leetcode | GeeksForGeeks kabirsk58694@gmail.com\nGithub | LinkedIn Kolkata, West Bengal'),
 Document(id='fef91087-f55e-4275-adff-2f2013340176', metadata={}, page_content='logs to trace user'),
 Document(id='cb90dcc7-5c3d-48cc-8a7f-7067be0b7488', metadata={}, page_content='– Designed and deployed ameta-agent orchestration layerwith aSupervisor Agentto manage multi-agent'),
 Document(id='57a00402-f092-4446-9e22-74ec3bd02091', metadata={}, page_content='∗ Built an intelligent agent framework usingLangGraph and integrated Groq-hosted Gemma-9B LLM,'),
 Document(id='ce9a07ed-2e7c-427e-b52d-3b8215f56dec', metadata={}, page_content='University of Engineering and Management, Kolkata\nEducation'),
 Document(id='03058977-a8f8-4856-9a73-07fbf9ee7a7c', metadata={},

In [147]:
vectorestore.add_documents(chunks)

['e7683329-368f-4bbc-8837-5bf36c595db1',
 '9c5ead9f-d79e-463b-bd71-e8204dc70bfc',
 'c08b9ccb-6f5e-4270-8825-904081256fc2',
 'd7c3458c-c999-4ad3-b4e5-9e676fbba26e',
 '038e2b10-7afc-421f-be1f-0950381fccd7',
 'd82c8d60-c858-4f26-ae31-174ac8889cfd',
 'fdd1afd3-e286-428e-9525-c841a844b507',
 '54c8eae5-cfa4-4a3c-becb-ba2336f33e01',
 'cc190666-596e-4084-9bbc-7c5018857b57',
 'a6bef9d2-c4ff-4367-a42b-04cf4096c6be',
 'e9808743-16f8-44e8-aafd-f9b1cddacd7d',
 'a5ef1c3e-c14a-4e7f-8c73-267afe3f93e7',
 '8754aaa3-f786-4579-9464-7bdc41b1867e',
 '650df324-f725-4139-8188-7914ff6ec581',
 '0e204cd8-f1f1-4d77-995e-c41395920642',
 'a0a3f6ea-fad1-43b3-9301-3a4760e9d6cf',
 'b342a9d2-5027-4a09-b080-688d401414c3',
 '1b5f0152-f5ad-4f0f-a362-cf5c81a84d96',
 'd9a2d225-9c4f-4b72-a4a9-b6aa9ff02c2d',
 '3718272e-afec-4ecf-9c7c-31e8379ccebf',
 'ba481fa5-bf61-436b-bcad-8eef1e766479',
 'e7ccbdba-30cb-4299-b22a-5db64044f0f2',
 'aa263f3b-8f63-45e9-aa9a-2a5014a524c1',
 'cef44726-7cba-4d86-8cb8-9e9a4dcf6770',
 '282bd3a0-1b8a-

In [148]:
vectorestore.similarity_search('what is the base compensation')

[Document(id='d7c3458c-c999-4ad3-b4e5-9e676fbba26e', metadata={'producer': 'Skia/PDF m123', 'creator': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36', 'creationdate': '2024-04-03T12:51:00+00:00', 'title': 'EMPLOYEE AGREEMENT', 'moddate': '2024-04-03T12:51:00+00:00', 'source': 'EMPLOYEE_AGREEMENT.pdf', 'total_pages': 13, 'page': 1, 'page_label': '2'}, page_content='ARTICLE 3\nPlace of Employment\n3.1. Place of Employment.   Employee shall perform his duties under this Employee Agreement at\n802 NW 5th Avenue, Suite 100, Gainesville FL 32601.\nARTICLE 4\nCompensation of Employee\n4.1. Base Compensation.  For all services rendered by Employee under this Employee Agreement, the\nCompany agrees to pay Employee the rate of $14,583 per month (the “base salary”), which shall be payable\nto Employee not less frequently than bi-monthly, or as is consistent with the Company’s practice for its other\nemployees.  \n4.2. Other Compen

In [ ]:
#